# Diagnóstico de cuadros y alfombras sobre fotos reales

El objetivo es localizar el error antes de volver a decorar. Comparamos la máscara del modelo con la anotación humana, dibujamos las esquinas sobre la foto original y solo después proyectamos la textura.

Los ejemplos se leen del ZIP oficial de ADE20K. Se han elegido para explicar fallos concretos. No constituyen una evaluación general del sistema. Las esquinas de diagnóstico y las referencias de perspectiva se han marcado a mano.

## 1. Preparar el entorno

Instala las dependencias desde la raíz con `python -m pip install -r requirements-photo.txt notebook matplotlib`. Se necesita el ZIP en `data/raw/ADEChallengeData2016.zip`. La primera carga de SegFormer descarga los pesos si aún no están en caché.

In [ ]:
from pathlib import Path
from zipfile import ZipFile
import inspect
import sys

import cv2 as cv
import numpy as np
import matplotlib.pyplot as plt

root = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "src/decorador_ar/photo.py").is_file())
sys.path.insert(0, str(root / "src"))
from decorador_ar.ade import ADEValidation
from decorador_ar.photo import Segmenter, iou, read_image, read_planes
from decorador_ar.placement import choose_quad, composite, default_asset, floor_direction

zip_path = root / "data/raw/ADEChallengeData2016.zip"
assert zip_path.is_file(), f"Falta {zip_path}"
segmenter = Segmenter()
print("Revisión del modelo:", segmenter.revision)

def read_pair(number):
    stem = f"ADE_val_{number:08d}"
    with ZipFile(zip_path) as archive:
        return ADEValidation(archive).read_pair(stem)

def show(ax, image, title):
    ax.imshow(cv.cvtColor(image, cv.COLOR_BGR2RGB) if image.ndim == 3 else image)
    ax.set_title(title)
    ax.axis("off")

def draw_segments(ax, segments):
    for points, color, label in segments:
        line = np.asarray(points)
        ax.plot(line[:, 0], line[:, 1], "o-", color=color, linewidth=2,
                markersize=3, label=label)
    ax.legend(loc="lower left", fontsize=8)

def closed(points):
    return np.vstack([points, points[0]])

## 2. ¿El modelo confundió la superficie?

En el dormitorio 117, el cuadro antiguo atraviesa la zona de la ventana. Azul indica pared correctamente detectada, rojo indica píxeles que el modelo llamó pared pero ADE20K anotó como otra clase, y amarillo indica pared que el modelo omitió. Los píxeles ADE sin etiqueta se excluyen.

In [ ]:
bedroom, bedroom_gt = read_pair(117)
wall, wall_conf = segmenter.predict(bedroom)["wall"]
old_corners = np.float32([[101, 149], [216, 149], [216, 227], [101, 227]])

def footprint(points, shape):
    return cv.fillConvexPoly(np.zeros(shape, np.uint8),
                             np.rint(points).astype(np.int32), 1).astype(bool)

def coverage(points, mask, valid=None):
    area = footprint(points, mask.shape)
    if valid is not None:
        area &= valid
    return float(mask[area].mean()) if area.any() else float("nan")

valid = bedroom_gt != 0
expected = bedroom_gt == 1
errors = (bedroom * .58).astype(np.uint8)
errors[wall & expected] = (220, 130, 25)
errors[wall & ~expected & valid] = (45, 60, 231)
errors[~wall & expected] = (20, 205, 240)

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
show(axes[0], bedroom, "Foto original")
show(axes[1], errors, "Predicción frente a anotación")
show(axes[2], wall_conf, "Confianza de pared")
draw_segments(axes[1], [(closed(old_corners), "white", "Cuadro antiguo")])
plt.show()
print(f"IoU de pared en toda la foto: {iou(wall, bedroom_gt, 1):.1%}")
print(f"Huellas sobre pared anotada: {coverage(old_corners, expected, valid):.1%}")

## 3. ¿La pieza está torcida aunque toque pared?

En el salón 522, el cuadro antiguo tiene el borde superior horizontal. El cuadro ya presente en la misma pared aporta una referencia visible de su inclinación. Sus bordes no tienen que ser exactamente paralelos en perspectiva, pero una diferencia importante merece inspección. También se ve si la ubicación compite con otra pieza.

Las dos líneas de referencia se marcaron visualmente. Esta comparación diagnostica el fallo, no estima la orientación completa de la cámara.

In [ ]:
room, room_gt = read_pair(522)
wrong_wall_quad = np.float32([[451, 237], [542, 237], [542, 310], [451, 310]])
reference_top = np.float32([[482, 125], [627, 99]])
reference_bottom = np.float32([[482, 225], [627, 204]])

def angle(points):
    dx, dy = np.asarray(points)[1] - np.asarray(points)[0]
    return np.degrees(np.arctan2(dy, dx))

fig, ax = plt.subplots(figsize=(10, 6))
show(ax, room, "Primero se comprueba el sitio con un contorno")
draw_segments(ax, [(closed(wrong_wall_quad), "orange", "Propuesta anterior"),
                   (reference_top, "cyan", "Borde superior del cuadro existente"),
                   (reference_bottom, "lime", "Borde inferior del cuadro existente")])
plt.show()
print(f"Borde propuesto: {angle(wrong_wall_quad[:2]):.1f}°")
print(f"Referencias de la pared: {angle(reference_top):.1f}° y {angle(reference_bottom):.1f}°")
print(f"Huella sobre pared anotada: {coverage(wrong_wall_quad, room_gt == 1, room_gt != 0):.1%}")

## 4. Leer la implementación que produjo el fallo

La máscara permite saber qué píxeles parecen de pared o suelo. En el modo experimental, la rama de pared fija un rectángulo horizontal. La rama de suelo busca líneas que convergen, con un valor aproximado si no encuentra suficientes. Ninguna de las dos ramas decide si es sensato añadir una segunda alfombra o decorar debajo de un cuadro existente.

In [ ]:
print("Segmentación")
print(inspect.getsource(Segmenter.predict))
print("Colocación experimental")
print(inspect.getsource(choose_quad))
print("Perspectiva aproximada del suelo")
print(inspect.getsource(floor_direction))

## 5. Ensayar las esquinas antes de pintar

Usamos el salón 1517 y una huella manual editable para la alfombra. Mira el contorno y la pata de la mesa antes de ver el montaje. Cambia las cuatro esquinas para comparar una huella demasiado pequeña, una demasiado torcida o una que cruza un mueble. Si el contorno ya se ve raro, el problema está en la colocación. Si solo se ve raro después de renderizar, revisa la textura y la proyección.

El orden de los puntos es superior izquierda, superior derecha, inferior derecha e inferior izquierda. La comodidad visual, la profundidad y la escala física siguen abiertas.

In [ ]:
rug_room, rug_gt = read_pair(1517)
rug_mask, rug_conf = segmenter.predict(rug_room)["floor"]
previous_guided = read_planes(root / "assets/planes/ADE_val_00001517.json",
                              rug_room.shape[1::-1])["floor"]
trial_quad = previous_guided.copy()  # Sustituye aquí las cuatro esquinas para comparar.
print("Esquinas:", trial_quad.tolist())
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
show(axes[0], rug_room, "Contorno sin textura")
draw_segments(axes[0], [(closed(trial_quad), "magenta", "Huella de la alfombra")])
show(axes[1], rug_mask, "Suelo predicho")
draw_segments(axes[1], [(closed(trial_quad), "magenta", "Mismas esquinas")])
plt.show()
supports = coverage(trial_quad, rug_mask)
print(f"Huella respaldada por suelo predicho: {supports:.1%}")
print(f"Huella sobre suelo anotado: {coverage(trial_quad, rug_gt == 4, rug_gt != 0):.1%}")

In [ ]:
rendered = rug_room.copy()
if supports >= .97:
    rug_texture = default_asset("floor", read_image)
    composite(rendered, rug_texture, trial_quad, rug_mask, "floor")
else:
    print("Render omitido: menos del 97 % de la huella está respaldado por la máscara.")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
show(axes[0], rug_room, "Original")
show(axes[1], rendered, "Prueba de proyección")
plt.show()
print(inspect.getsource(composite))

## 6. Qué queda por resolver

1. **Superficie y obstáculos.** Medir qué confunde el modelo y excluir ventanas, muebles y alfombras existentes. La anotación ADE es referencia de evaluación.
2. **Orientación.** Estimar la dirección de cada pared y del suelo a partir de líneas fiables. Mostrar sobre la foto qué líneas se usaron. Si faltan líneas, permitir marcar la superficie.
3. **Ubicación y escala.** Evaluar las esquinas dibujadas antes de gastar tiempo en el render. Una colocación necesita paso libre, separación de otros objetos y tamaño creíble. Una sola foto sin referencia no aporta escala física.
4. **Acabado.** Comparar fotos originales y montajes sin seleccionar solo los casos favorables. Anotar si el fallo es superficie, ocupación, perspectiva, escala o composición.